# TF-b · GradientTape theo tọa độ

Mục tiêu: hiểu tape.watch, đạo hàm bậc cao và phân biệt gradient theo x với gradient theo trọng số.


In [ ]:
import tensorflow as tf
tf.keras.backend.set_floatx('float64')
print('TensorFlow', tf.__version__)


## 1. Vì sao phải watch(x)?

Tensor hằng không được tape theo dõi tự động. Tape phải thấy phép tính y=x³ sau khi watch x.


In [ ]:
x = tf.constant([[0.5], [1.0]], dtype=tf.float64)
with tf.GradientTape() as tape:
    tape.watch(x)
    y = x**3
dy = tape.gradient(y, x)
print(dy.numpy().reshape(-1))  # 3*x**2: [0.75, 3.0]


## 2. Đạo hàm của đạo hàm

Tape ngoài phải bao phép lấy đạo hàm của tape trong. Mỗi tape dùng một lần, nên không cần persistent=True.


In [ ]:
x = tf.constant([[0.25], [0.75]], dtype=tf.float64)
with tf.GradientTape() as t4:
    t4.watch(x)
    with tf.GradientTape() as t3:
        t3.watch(x)
        with tf.GradientTape() as t2:
            t2.watch(x)
            with tf.GradientTape() as t1:
                t1.watch(x)
                p = x**4 - 2*x**3 + x
            p1 = t1.gradient(p, x)
        p2 = t2.gradient(p1, x)
    p3 = t3.gradient(p2, x)
p4 = t4.gradient(p3, x)
print(p4.numpy().reshape(-1))  # [24, 24]


## 3. Một lỗi có chủ ý

Không watch x thì gradient theo x là None. Tự bỏ tape.watch(x) trong một ô bản sao để kiểm tra, rồi khôi phục. Các trọng số Keras là tf.Variable nên được tape theo dõi tự động khi train.

Tự làm: thay p bằng x**4 và dự đoán p4. Giải thích vì sao tape ngoài phải bao tape trong.
